# Landslide combined class step 04: charts (maximum scenario)

Creates charts from the combined-class maximum-scenario summaries produced by step 03.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
output_path = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_maximum_scenario_combined_class'
damage_estimates_path = output_path / 'damage_estimates'

sector_scenario_summary_file = damage_estimates_path / 'sector_return_period_scenario_damages_usd_combined_class.csv'

print('Output path:', output_path)
print('Sector scenario summary file:', sector_scenario_summary_file)


In [ ]:
if not sector_scenario_summary_file.exists():
    raise FileNotFoundError(
        f'Missing summary file: {sector_scenario_summary_file}. Run the combined-class step 03 notebook first.'
    )

sector_pivot = pd.read_csv(sector_scenario_summary_file)

scenario_order = ['baseline', 'deforestation', 'reafforestation']
required_columns = {'Sector', 'ReturnPeriod', 'Avoided_Protection_USD', 'Avoided_Reafforestation_USD', *scenario_order}
missing_columns = required_columns.difference(sector_pivot.columns)
if missing_columns:
    raise KeyError(f'Missing required columns in {sector_scenario_summary_file.name}: {sorted(missing_columns)}')

sector_pivot['ReturnPeriod'] = pd.to_numeric(sector_pivot['ReturnPeriod'], errors='coerce').astype(int)
for scenario_name in scenario_order:
    sector_pivot[scenario_name] = pd.to_numeric(sector_pivot[scenario_name], errors='coerce').fillna(0.0)

print('Rows:', len(sector_pivot))
sector_pivot.head(20)


In [ ]:
# Total USD direct damages by scenario and return period across all sectors
pivot_total = sector_pivot.groupby('ReturnPeriod')[scenario_order].sum().sort_index()

pivot_total


In [ ]:
# Grouped bar chart: direct damages (USD) by return period and scenario
return_periods = pivot_total.index.to_list()
bar_positions = np.arange(len(return_periods))
bar_width = 0.24

scenario_colors = {
    'baseline': '#4C78A8',
    'deforestation': '#F58518',
    'reafforestation': '#54A24B',
}

figure, plot_axis = plt.subplots(figsize=(10, 6))

for scenario_index, scenario in enumerate(scenario_order):
    if scenario not in pivot_total.columns:
        continue

    offsets = bar_positions + (scenario_index - 1) * bar_width
    values = pivot_total[scenario].to_numpy()

    plot_axis.bar(
        offsets,
        values,
        width=bar_width,
        label=scenario.capitalize(),
        color=scenario_colors.get(scenario, '#888888'),
        edgecolor='white',
        linewidth=0.8,
    )

plot_axis.set_xticks(bar_positions)
plot_axis.set_xticklabels([str(return_period) for return_period in return_periods])
plot_axis.set_xlabel('Return period (years)')
plot_axis.set_ylabel('Direct damages (USD)')
plot_axis.set_title('Landslide Combined-Class Direct Damages in USD by Return Period and Scenario')
plot_axis.grid(axis='y', alpha=0.25)
plot_axis.legend(title='Scenario', frameon=True)

plot_axis.yaxis.set_major_formatter(FuncFormatter(lambda value, tick_position: f'${value:,.0f}'))

plt.tight_layout()

chart_file = damage_estimates_path / 'landslide_step04_direct_damages_usd_by_return_period_and_scenario_combined_class.png'
figure.savefig(chart_file, dpi=300, bbox_inches='tight')
print('Saved chart:', chart_file)
plt.show()


In [ ]:
# Sector-by-sector USD table produced by step 03
sector_pivot = sector_pivot.sort_values(['Sector', 'ReturnPeriod']).reset_index(drop=True)

sector_pivot.head(30)


In [ ]:
# RP-sector charts: direct damages and avoided damages
sector_order = ['buildings', 'transport', 'water', 'energy']
existing_sectors = [sector_name for sector_name in sector_order if sector_name in sector_pivot['Sector'].unique()]
remaining_sectors = sorted([sector_name for sector_name in sector_pivot['Sector'].unique() if sector_name not in existing_sectors])
plot_sectors = existing_sectors + remaining_sectors

if not plot_sectors:
    raise ValueError('No sectors found in sector_pivot for plotting.')

# Ensure deterministic RP order
return_periods = sorted(sector_pivot['ReturnPeriod'].dropna().unique().tolist())

scenario_styles = {
    'baseline': {'color': '#4C78A8', 'label': 'Baseline'},
    'deforestation': {'color': '#F58518', 'label': 'Deforestation'},
    'reafforestation': {'color': '#54A24B', 'label': 'Reafforestation'},
}

# 1) Direct damages by RP across sectors
number_of_plot_sectors = len(plot_sectors)
subplot_columns = 2
subplot_rows = int(np.ceil(number_of_plot_sectors / subplot_columns))
figure, plot_axes = plt.subplots(subplot_rows, subplot_columns, figsize=(13, 4.1 * subplot_rows), sharex=True)
plot_axes = np.array(plot_axes).reshape(-1)

for sector_index, sector_name in enumerate(plot_sectors):
    plot_axis = plot_axes[sector_index]
    sector_data = sector_pivot[sector_pivot['Sector'] == sector_name].sort_values('ReturnPeriod')

    for scenario in ['baseline', 'deforestation', 'reafforestation']:
        if scenario not in sector_data.columns:
            continue
        plot_axis.plot(
            sector_data['ReturnPeriod'],
            sector_data[scenario],
            marker='o',
            linewidth=2,
            markersize=4,
            color=scenario_styles[scenario]['color'],
            label=scenario_styles[scenario]['label'],
        )

    plot_axis.set_title(sector_name.capitalize())
    plot_axis.grid(alpha=0.25)
    plot_axis.set_xticks(return_periods)
    plot_axis.set_xlabel('Return period (years)')
    plot_axis.set_ylabel('Direct damages (USD)')
    plot_axis.yaxis.set_major_formatter(FuncFormatter(lambda value, tick_position: f'${value/1e6:,.0f}m'))

# Hide unused axes
for unused_axis_index in range(number_of_plot_sectors, len(plot_axes)):
    plot_axes[unused_axis_index].set_visible(False)

# Single shared legend
legend_handles, legend_labels = plot_axes[0].get_legend_handles_labels()
figure.legend(legend_handles, legend_labels, loc='upper center', ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.02))
figure.suptitle('Landslide Combined-Class Direct Damages by Return Period Across Sectors', y=1.05)
plt.tight_layout()

direct_sector_chart = damage_estimates_path / 'landslide_step04_sector_direct_damages_by_rp_and_scenario_combined_class.png'
figure.savefig(direct_sector_chart, dpi=300, bbox_inches='tight')
print('Saved chart:', direct_sector_chart)
plt.show()

# 2) Avoided damages by RP across sectors
sector_avoided_damage_data = sector_pivot.copy()

figure, plot_axes = plt.subplots(subplot_rows, subplot_columns, figsize=(13, 4.1 * subplot_rows), sharex=True)
plot_axes = np.array(plot_axes).reshape(-1)

for sector_index, sector_name in enumerate(plot_sectors):
    plot_axis = plot_axes[sector_index]
    sector_data = sector_avoided_damage_data[sector_avoided_damage_data['Sector'] == sector_name].sort_values('ReturnPeriod')

    plot_axis.plot(
        sector_data['ReturnPeriod'],
        sector_data['Avoided_Reafforestation_USD'],
        marker='o',
        linewidth=2,
        markersize=4,
        color='#2E7D32',
        label='Avoided via reafforestation (Baseline - Reafforestation)',
    )
    plot_axis.plot(
        sector_data['ReturnPeriod'],
        sector_data['Avoided_Protection_USD'],
        marker='o',
        linewidth=2,
        markersize=4,
        color='#EF6C00',
        label='Avoided via forest protection (Deforestation - Baseline)',
    )

    plot_axis.axhline(0.0, color='#777777', linewidth=1, linestyle='--')
    plot_axis.set_title(sector_name.capitalize())
    plot_axis.grid(alpha=0.25)
    plot_axis.set_xticks(return_periods)
    plot_axis.set_xlabel('Return period (years)')
    plot_axis.set_ylabel('Avoided damages (USD)')
    plot_axis.yaxis.set_major_formatter(FuncFormatter(lambda value, tick_position: f'${value/1e6:,.0f}m'))

for unused_axis_index in range(number_of_plot_sectors, len(plot_axes)):
    plot_axes[unused_axis_index].set_visible(False)

legend_handles, legend_labels = plot_axes[0].get_legend_handles_labels()
figure.legend(legend_handles, legend_labels, loc='upper center', ncol=1, frameon=True, bbox_to_anchor=(0.5, 1.06))
figure.suptitle('Landslide Combined-Class Avoided Damages by Return Period Across Sectors', y=1.10)
plt.tight_layout()

avoided_sector_chart = damage_estimates_path / 'landslide_step04_sector_avoided_damages_by_rp_combined_class.png'
figure.savefig(avoided_sector_chart, dpi=300, bbox_inches='tight')
print('Saved chart:', avoided_sector_chart)
plt.show()

sector_avoided_damage_data[['Sector', 'ReturnPeriod', 'Avoided_Reafforestation_USD', 'Avoided_Protection_USD', 'Combined_Benefit_USD']].head(20)